# KAM-CoT: Knowledge Augmented Multimodal Chain-of-Thoughts

**Google Colab Notebook — T4 GPU**

Kiến trúc:
- Language Encoder: FLAN-T5-Base (d=768)
- Vision Encoder: DETR / ViT + projection
- Graph Encoder: RGAT + GCN (edge embeddings 34×64)
- Cross-Attention: Lang→Image, Lang→KG
- Gated Fusion: Fusion-1 (W₁..W₉)
- Decoder: FLAN-T5 decoder

Các module nhỏ nằm trong thư mục `kam_cot/`, dễ dàng theo dõi và thay đổi tham số.

---
## 1. Cài đặt Dependencies

In [ ]:
# ============================================================
# Cài đặt thư viện cần thiết
# ============================================================
print("Installing dependencies...")

# Core — Colab T4 thường có sẵn torch 2.x với CUDA 12.x
# Không cần cài lại torch nếu đã có; chỉ cài khi cần override
# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# HuggingFace
!pip install -q transformers accelerate sentencepiece

# PyTorch Geometric — tự detect torch version và cài đúng backend
import torch as _torch
_tv = _torch.__version__.split('+')[0]   # e.g. '2.3.0'
_cu = 'cu' + _torch.version.cuda.replace('.', '')[:3] if _torch.cuda.is_available() else 'cpu'
print(f"Detected: torch={_tv}, cuda={_cu}")

# Cài torch-scatter / torch-sparse trước (optional, giúp PyG nhanh hơn)
# !pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{_tv}+{_cu}.html

# torch_geometric (pure-Python install, luôn tương thích)
!pip install -q torch_geometric

# Cài thêm pyg-lib để có RGATConv stable (optional)
# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyg-lib',
#                 '-f', f'https://data.pyg.org/whl/torch-{_tv}+{_cu}.html'])

# Optional: conceptnet-lite (cho KG thật)
!pip install -q conceptnet-lite

# Image & utils
!pip install -q pillow requests tqdm

print("Done!")

# Kiểm tra GPU
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Kiểm tra PyG version
import torch_geometric
print(f"PyG version: {torch_geometric.__version__}")


---
## 2. Mount Google Drive & Upload Code

**Cách 1 (recommended):** Upload thư mục `kam_cot/` lên Drive, mount và copy vào Colab.

**Cách 2:** Copy trực tiếp các file .py vào Colab runtime.

In [ ]:
# === Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === Copy kam_cot module từ Drive vào Colab ===
import shutil, os

# Đường dẫn đến thư mục kam_cot trên Drive (cần upload trước)
DRIVE_SRC = "/content/drive/MyDrive/kam_cot"  # THAY ĐỔI NẾU CẦN
COLAB_DST = "/content/kam_cot"

if os.path.exists(DRIVE_SRC):
    if os.path.exists(COLAB_DST):
        shutil.rmtree(COLAB_DST)
    shutil.copytree(DRIVE_SRC, COLAB_DST)
    print(f"Copied kam_cot from Drive to {COLAB_DST}")
elif os.path.exists("/content/kam_cot/__init__.py"):
    print("kam_cot already in /content/")
else:
    print(f"[WARN] {DRIVE_SRC} not found!")
    print("Upload kam_cot/ folder to Drive or use the upload button in Colab.")

In [ ]:
# Kiểm tra module đã sẵn sàng
import sys
sys.path.append('/content')

try:
    from kam_cot import (
        CrossAttention, GatedFusion, VisionEncoder,
        GraphEncoder, ConceptNetExtractor, KAMCoTModel
    )
    print("KAM-CoT modules loaded successfully!")
    print(f"  CrossAttention: {CrossAttention.__module__}")
    print(f"  GatedFusion: {GatedFusion.__module__}")
    print(f"  VisionEncoder: {VisionEncoder.__module__}")
    print(f"  GraphEncoder: {GraphEncoder.__module__}")
    print(f"  ConceptNetExtractor: {ConceptNetExtractor.__module__}")
    print(f"  KAMCoTModel: {KAMCoTModel.__module__}")
except ImportError as e:
    print(f"Module import failed: {e}")
    print("Upload the kam_cot/ folder first.")

---
## 3. Khởi tạo Model

**Các tham số có thể thay đổi tại đây:**

In [ ]:
from transformers import T5Tokenizer, AutoImageProcessor
from kam_cot import KAMCoTModel

# ============================================================
#  CẤU HÌNH MODEL — Có thể thay đổi thoải mái
# ============================================================
MODEL_CONFIG = {
    # Language backbone
    "model_name": "google/flan-t5-base",  # FLAN-T5-Base (280M)
    # "model_name": "google/flan-t5-small",   # FLAN-T5-Small (80M) — nhẹ hơn
    # "model_name": "google/flan-t5-large",   # FLAN-T5-Large (780M) — nặng hơn

    # Vision backbone
    "vision_model": "facebook/detr-resnet-101-dc5",  # DETR (mặc định)
    # "vision_model": "google/vit-base-patch16-384",  # ViT (dùng với use_vit=True)
    "use_vit": False,   # True = ViT, False = DETR

    # Graph encoder
    "graph_edge_dim": 64,     # kích thước edge embedding
    "graph_num_rels": 34,     # số loại quan hệ có hướng
    "graph_num_heads": 4,     # số head trong RGAT

    # Regularization
    "dropout": 0.1,
}

print("Initializing KAM-CoT model...")
print(f"  Language: {MODEL_CONFIG['model_name']}")
print(f"  Vision:   {'ViT' if MODEL_CONFIG['use_vit'] else 'DETR'} ({MODEL_CONFIG['vision_model']})")

model = KAMCoTModel(**MODEL_CONFIG)
tokenizer = T5Tokenizer.from_pretrained(MODEL_CONFIG['model_name'])

# Image processor
if MODEL_CONFIG['use_vit']:
    image_processor = AutoImageProcessor.from_pretrained(MODEL_CONFIG['vision_model'])
else:
    from transformers import DetrImageProcessor
    image_processor = DetrImageProcessor.from_pretrained(MODEL_CONFIG['vision_model'])

# Đếm tham số
params = model.count_parameters()
print(f"\nModel parameters:")
print(f"  Total:    {params['total'] / 1e6:.2f}M")
print(f"  Trainable: {params['trainable'] / 1e6:.2f}M")

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"  Device: {device}")

---
## 4. Sanity Check — Forward Pass

Kiểm tra model chạy được với input giả (synthetic).

In [ ]:
import torch
import numpy as np

print("Running sanity check...")

# === Synthetic Input ===
batch_size = 2
text_len = 32
target_len = 16
d_model = model.d_model  # 768
max_nodes = 10

# Lấy device từ model để đặt tensor đúng chỗ
device = next(model.parameters()).device

# Text
input_ids = torch.randint(0, 32100, (batch_size, text_len), device=device)
attention_mask = torch.ones(batch_size, text_len, dtype=torch.long, device=device)
labels = torch.randint(0, 32100, (batch_size, target_len), device=device)

# Image (giả DETR output: 3×480×640)
pixel_values = torch.randn(batch_size, 3, 480, 640, device=device)

# KG
kg_node_features = torch.randn(batch_size * max_nodes, d_model, device=device)
# Edge index (COO)
num_edges = 20
kg_edge_index = torch.randint(0, max_nodes, (2, num_edges), device=device)
kg_edge_type = torch.randint(0, 34, (num_edges,), device=device)

print(f"  Batch size: {batch_size}")
print(f"  Text length: {text_len}")
print(f"  Image shape: {tuple(pixel_values.shape)}")
print(f"  KG nodes: {max_nodes} per sample, edges: {num_edges}")

# === Forward ===
model.eval()
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        kg_node_features=kg_node_features,
        kg_edge_index=kg_edge_index,
        kg_edge_type=kg_edge_type,
        labels=labels,
    )

print(f"\n  Forward pass successful!")
print(f"  Loss: {outputs['loss'].item():.4f}")
print(f"  Logits shape: {tuple(outputs['logits'].shape)}")
print(f"  H_fuse shape: {tuple(outputs['H_fuse'].shape)}")
print(f"  H_lang shape: {tuple(outputs['H_lang'].shape)}")
print(f"  H_img shape: {tuple(outputs['H_img'].shape)}")
print(f"  H_kg shape: {tuple(outputs['H_kg'].shape)}")

# === Generate ===
print("\nTesting generation...")
with torch.no_grad():
    gen_ids = model.generate(
        input_ids=input_ids[:1],
        attention_mask=attention_mask[:1],
        pixel_values=pixel_values[:1],
        kg_node_features=kg_node_features[:max_nodes],
        kg_edge_index=kg_edge_index,
        kg_edge_type=kg_edge_type,
        max_length=20,
        num_beams=2,
    )
print(f"  Generation output shape: {tuple(gen_ids.shape)}")
print(f"  Generated tokens: {gen_ids.tolist()}")

print("\n✓ Model hoạt động bình thường!")

---
## 5. Kiểm tra Knowledge Graph Extractor

Trích xuất đồ thị tri thức từ câu hỏi mẫu.
Dùng conceptnet-lite nếu đã cài, fallback về subgraph đơn giản.

In [ ]:
from kam_cot import ConceptNetExtractor

# Khởi tạo KG extractor
kg_extractor = ConceptNetExtractor(
    language='en',
    max_nodes=200,
    max_hops=2,
)

# Hàm nhúng node (dùng T5 token embeddings)
node_embed_fn = model.get_node_embed_fn(tokenizer)

# Test với câu hỏi mẫu
sample_question = "What is the capital of France?"
print(f"Question: {sample_question}")

# Trích xuất entities
entities = kg_extractor.extract_entities(sample_question)
print(f"Extracted entities: {entities}")

# Build graph
kg_data = kg_extractor.build_graph_data(sample_question, node_embed_fn)
print(f"\nKG data:")
print(f"  Node features: {tuple(kg_data['node_features'].shape)}")
print(f"  Edge index: {tuple(kg_data['edge_index'].shape)}")
print(f"  Edge types: {tuple(kg_data['edge_type'].shape)}")
print(f"  Num edges: {kg_data['edge_index'].size(1)}")

---
## 6. Demo đầy đủ

Chạy model với input đầy đủ (text + image + KG).

In [ ]:
from PIL import Image
import requests

# === Tạo câu hỏi mẫu ===
question = (
    "Context: A cat is sitting on a mat. "
    "Question: What is the cat sitting on? "
    "Choices: (A) mat (B) table (C) chair"
)

print(f"Input text:\n  {question}\n")

# === Tokenize ===
inputs = tokenizer(
    question,
    max_length=512,
    truncation=True,
    padding='max_length',
    return_tensors='pt',
)

# === Tạo ảnh giả (hoặc dùng ảnh thật từ URL) ===
try:
    url = "http://images.cocodataset.org/val2017/000000039769.jpg"  # cats image
    img = Image.open(requests.get(url, stream=True).raw).convert('RGB')
    print(f"Image loaded: {img.size}")
    if MODEL_CONFIG['use_vit']:
        img_tensor = image_processor(img, return_tensors='pt')['pixel_values']
    else:
        img_tensor = image_processor(img, return_tensors='pt')['pixel_values']
except Exception as e:
    print(f"Image load failed: {e} — using random tensor")
    img_tensor = torch.randn(1, 3, 480, 640)

# === Knowledge Graph ===
kg_data = kg_extractor.build_graph_data(question, node_embed_fn)

# === Generate ===
device = next(model.parameters()).device
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs['input_ids'].to(device),
        attention_mask=inputs['attention_mask'].to(device),
        pixel_values=img_tensor.to(device),
        kg_node_features=kg_data['node_features'].to(device),
        kg_edge_index=kg_data['edge_index'].to(device),
        kg_edge_type=kg_data['edge_type'].to(device),
        max_length=64,
        num_beams=4,
    )

output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n=== Model Output ===")
print(f"  {output_text}")

---
## 7. Training (2-Stage)

**Quan trọng:** Cần có dataset với các trường:
- `question`: câu hỏi
- `context`: ngữ cảnh (optional)
- `choices`: các lựa chọn (optional)
- `rationale`: chuỗi suy luận (Stage 1)
- `answer`: đáp án (Stage 2)
- `image_path`: đường dẫn ảnh (optional)

Dưới đây là code mẫu với synthetic data để kiểm tra pipeline.

In [ ]:
from kam_cot.training import (
    TrainingConfig, KAMCoTDataset, KAMCoTTrainer,
    setup_stage1_training, setup_stage2_training
)

# ============================================================
#  CẤU HÌNH TRAINING — Thay đổi thoải mái
# ============================================================
TRAIN_CONFIG = {
    # Dataloader
    "batch_size": 2,                    # Giảm nếu OOM (T4 16GB)
    "gradient_accumulation_steps": 4,   # Batch ảo = batch_size * accumulation
    "max_text_length": 256,
    "max_target_length": 64,
    "max_nodes": 50,                    # Giảm để tiết kiệm VRAM

    # Optimization
    "learning_rate": 3e-5,
    "warmup_steps": 20,
    "num_epochs": 3,                    # Tăng khi có data thật
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,

    # Hardware
    "fp16": True,                       # Mixed precision (tiết kiệm VRAM)

    # Logging
    "logging_steps": 5,
    "output_dir": "./kam_cot_output",
}

print("Training config ready.")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================
#  Tạo synthetic data để test pipeline
# ============================================================

synthetic_data = []
qa_pairs = [
    {
        "question": "What is the capital of France?",
        "context": "France is a country in Europe.",
        "choices": ["London", "Paris", "Berlin", "Madrid"],
        "rationale": "France is a country in Europe. The capital of France is Paris.",
        "answer": "Paris",
    },
    {
        "question": "What does a cat eat?",
        "context": "Cats are domestic animals.",
        "choices": ["Fish", "Bones", "Grass", "Stones"],
        "rationale": "Cats are carnivores and like to eat fish.",
        "answer": "Fish",
    },
    {
        "question": "What is the largest planet?",
        "context": "The solar system has eight planets.",
        "choices": ["Earth", "Mars", "Jupiter", "Saturn"],
        "rationale": "Jupiter is the largest planet in the solar system.",
        "answer": "Jupiter",
    },
    {
        "question": "What is water made of?",
        "context": "Water is essential for life.",
        "choices": ["Hydrogen and Oxygen", "Salt", "Sugar", "Carbon"],
        "rationale": "Water is H2O, composed of hydrogen and oxygen.",
        "answer": "Hydrogen and Oxygen",
    },
    {
        "question": "Which animal can fly?",
        "context": "Many animals live on Earth.",
        "choices": ["Dog", "Cat", "Bird", "Fish"],
        "rationale": "Birds have wings and can fly.",
        "answer": "Bird",
    },
]

for qa in qa_pairs:
    synthetic_data.append(qa)
    # Thêm 1 bản copy với image_path giả
    synthetic_data.append({**qa, "image_path": None})

print(f"Created {len(synthetic_data)} synthetic samples")
print(f"\nSample:")
print(f"  Q: {synthetic_data[0]['question']}")
print(f"  Rationale: {synthetic_data[0]['rationale']}")
print(f"  Answer: {synthetic_data[0]['answer']}")

In [ ]:
# ============================================================
#  STAGE 1: Rationale Generation
# ============================================================

stage1_config = TrainingConfig(
    stage=1,
    freeze_encoder=False,
    **TRAIN_CONFIG,
)

trainer1 = setup_stage1_training(
    model=model,
    tokenizer=tokenizer,
    train_data=synthetic_data,
    eval_data=synthetic_data[:2],
    config=stage1_config,
    kg_extractor=kg_extractor,
)

print(f"Stage 1 trainer ready.")
print(f"  Train samples: {len(trainer1.train_dataset)}")
print(f"  Eval samples: {len(trainer1.eval_dataset) if trainer1.eval_dataset else 0}")
print(f"  Device: {trainer1.device}")

# === BẮT ĐẦU TRAINING ===
results1 = trainer1.train()

In [ ]:
# ============================================================
#  STAGE 2: Answer Prediction
#  (freeze encoder, chỉ train decoder)
# ============================================================

stage2_config = TrainingConfig(
    stage=2,
    freeze_encoder=True,
    **TRAIN_CONFIG,
)

trainer2 = setup_stage2_training(
    model=model,
    tokenizer=tokenizer,
    train_data=synthetic_data,
    eval_data=synthetic_data[:2],
    config=stage2_config,
    kg_extractor=kg_extractor,
    freeze_encoder=True,
)

print(f"Stage 2 trainer ready.")

# === BẮT ĐẦU TRAINING ===
results2 = trainer2.train()

In [ ]:
# ============================================================
#  Kiểm tra model sau training
# ============================================================

test_question = "Question: What is the capital of France? Choices: (A) London (B) Paris (C) Berlin"

inputs = tokenizer(
    test_question,
    max_length=256, truncation=True,
    padding='max_length', return_tensors='pt',
).to(device)

kg_data = kg_extractor.build_graph_data(test_question, node_embed_fn)

model.eval()
with torch.no_grad():
    gen_ids = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        kg_node_features=kg_data['node_features'].to(device),
        kg_edge_index=kg_data['edge_index'].to(device),
        kg_edge_type=kg_data['edge_type'].to(device),
        max_length=64,
        num_beams=4,
    )

output = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
print(f"Input: {test_question}")
print(f"Output: {output}")

---
## 8. Lưu Model

Lưu checkpoint về Drive để dùng sau.

In [ ]:
# Lưu về Drive
save_path = "/content/drive/MyDrive/kam_cot_checkpoint"
os.makedirs(save_path, exist_ok=True)

torch.save(model.state_dict(), f"{save_path}/kam_cot_model.pt")
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")
print(f"  File: kam_cot_model.pt")
print(f"  Size: {os.path.getsize(f'{save_path}/kam_cot_model.pt') / 1e6:.1f} MB")

---
## Tổng kết

**Các module trong thư mục `kam_cot/`:**

| File | Chức năng |
|------|-----------|
| `cross_attention.py` | Cross-Attention đơn đầu (Lang→Img, Lang→KG) |
| `gated_fusion.py` | Gated Fusion với W₁..W₉ |
| `vision_encoder.py` | Vision Encoder (DETR / ViT) |
| `image_captioner.py` | Image Captioning bổ trợ (ViT-GPT2) |
| `graph_encoder.py` | Graph Encoder (RGAT + GCN) |
| `graph_extractor.py` | ConceptNet extractor (KG) |
| `model.py` | KAM-CoT model tổng hợp |
| `training.py` | Training pipeline (2-Stage) |

**Để thay đổi tham số:** sửa trực tiếp ở Cell 3 (MODEL_CONFIG) và Cell 7 (TRAIN_CONFIG).

**Lưu ý T4 (16GB VRAM):**
- FLAN-T5-Base (~280M) + DETR (~60M) ≈ 340M params → ~4-6GB VRAM
- Batch size 2-4 với gradient accumulation
- Bật FP16 mixed precision (mặc định) để tiết kiệm VRAM
- Giảm `max_nodes` và `max_text_length` nếu OOM